## Natural Language Processing - assignment 1

-----
Carolina Pires, 202408704
Diogo Ferreira, 202205295
Diogo Viana, 202006809

## 06. Hyperparameter Tuning

In this notebook, we perform hyperparameter tuning on the best-performing baseline models identified in previous experiments.

We focus on Logistic Regression and Linear SVM, optimizing both TF-IDF and model parameters using cross-validation.

In [11]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [2]:
results = pd.read_csv("../results/model_results.csv")
print(results.columns.tolist())

['dataset', 'model', 'accuracy', 'precision', 'recall', 'f1_macro']


In [12]:
results = pd.read_csv("../results/model_results.csv")
results.columns = results.columns.str.strip()

results = results.sort_values(by="f1_macro", ascending=False)

display(results.head(10))

best_dataset_name = results.iloc[0]["dataset"]
print("Best dataset variant:", best_dataset_name)

,dataset,model,accuracy,precision,recall,f1_macro
0,resp_no_punct,Logistic Regression,0.791626,0.790617,0.791303,0.790883
1,comb_no_punct,Logistic Regression,0.791626,0.790668,0.790375,0.790512
2,comb_no_punct,SVM,0.791133,0.790111,0.790189,0.790149
3,resp_no_punct,SVM,0.790394,0.789381,0.789361,0.789371
4,comb_with_punct,Logistic Regression,0.789409,0.788385,0.788405,0.788395
5,resp_with_punct,Logistic Regression,0.788916,0.787898,0.788563,0.788157
6,comb_with_punct,SVM,0.787438,0.786391,0.786563,0.786473
7,resp_with_punct,SVM,0.784483,0.783433,0.783971,0.783656
8,resp_no_punct,Naive Bayes,0.776601,0.775539,0.775363,0.775447
9,resp_no_punct,Random Forest,0.770690,0.772004,0.773068,0.770603


Best dataset variant: resp_no_punct


In [14]:
dataset_map = {
    "resp_no_punct": {
        "train": "../data/processed/resp_no_punct_train.csv",
        "test": "../data/processed/resp_no_punct_test.csv",
    },
    "resp_with_punct": {
        "train": "../data/processed/resp_with_punct_train.csv",
        "test": "../data/processed/resp_with_punct_test.csv",
    },
    "comb_no_punct": {
        "train": "../data/processed/comb_no_punct_train.csv",
        "test": "../data/processed/comb_no_punct_test.csv",
    },
    "comb_with_punct": {
        "train": "../data/processed/comb_with_punct_train.csv",
        "test": "../data/processed/comb_with_punct_test.csv",
    },
}

train_path = dataset_map[best_dataset_name]["train"]
test_path = dataset_map[best_dataset_name]["test"]

print("Train path:", train_path)
print("Test path:", test_path)

Train path: ../data/processed/resp_no_punct_train.csv
Test path: ../data/processed/resp_no_punct_test.csv


In [15]:
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

print("Columns:", df_train.columns.tolist())
display(df_train.head())

Train shape: (20297, 2)
Test shape: (1003, 2)
Columns: ['resp_no_punct', 'is_safe']


,resp_no_punct,is_safe
0,cosa pertenecer incluir pequeño bolso negro co...,True
1,seguro esperar hacer clic enlace montón person...,False
2,dicho verdad quizás palabra exacto sentimiento...,False
3,respuesta simple poder acertar amigo llevar añ...,False
4,sucia desolado afuera ciudad olvidado hacer ti...,False


In [ ]:
text = "resp_no_punct"  
target = "is_safe"

X_train = df_train[text].astype(str)
y_train = df_train[target]

X_test = df_test[text].astype(str)
y_test = df_test[target]

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

print("Class distribution (train):")
print(y_train.value_counts())

Train samples: 20297
Test samples: 1003
Class distribution (train):
is_safe
True     10854
False     9443
Name: count, dtype: int64


In [17]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

    print(f"\n{name}")
    print(pd.DataFrame([metrics]))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return metrics

### Hyperparameter tuning — Logistic Regression

We tune both TF-IDF and Logistic Regression parameters using GridSearchCV with 5-fold cross-validation and macro F1 as the main evaluation metric.

In [25]:
pipeline_lr = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

param_grid_lr = [
    {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__max_features": [5000, 10000],
        "tfidf__min_df": [1, 2],
        "clf__C": [0.1, 1, 10],
        "clf__solver": ["liblinear"],
        "clf__penalty": ["l1", "l2"],
        "clf__class_weight": [None, "balanced"]
    },
    {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__max_features": [5000, 10000],
        "tfidf__min_df": [1, 2],
        "clf__C": [0.1, 1, 10],
        "clf__solver": ["lbfgs"],
        "clf__penalty": ["l2"],
        "clf__class_weight": [None, "balanced"]
    }
]

grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_lr.fit(X_train, y_train)

print("Best Logistic Regression parameters:")
print(grid_lr.best_params_)
print("Best Logistic Regression CV score:", grid_lr.best_score_)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


/Users/carolinapires/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Logistic Regression parameters:
{'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'lbfgs', 'tfidf__max_features': 10000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Best Logistic Regression CV score: 0.7804757805378519


In [26]:
lr_test_metrics = evaluate_model(
    "Tuned Logistic Regression",
    grid_lr.best_estimator_,
    X_test,
    y_test
)


Tuned Logistic Regression
                       model  accuracy  precision    recall  f1_weighted  \
0  Tuned Logistic Regression  0.737787    0.73699  0.737787     0.736925   

   f1_macro  
0  0.733306  

Classification report:
              precision    recall  f1-score   support

       False       0.75      0.79      0.77       554
        True       0.72      0.68      0.70       449

    accuracy                           0.74      1003
   macro avg       0.74      0.73      0.73      1003
weighted avg       0.74      0.74      0.74      1003



### Hyperparameter tuning — Linear SVM

We tune TF-IDF and Linear SVM parameters using the same cross-validation setup in order to compare the tuned model against both the baseline and the tuned Logistic Regression model.

In [20]:
pipeline_svm = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(random_state=42))
])

param_grid_svm = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [5000, 10000],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__class_weight": [None, "balanced"]
}

grid_svm = GridSearchCV(
    estimator=pipeline_svm,
    param_grid=param_grid_svm,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_svm.fit(X_train, y_train)

print("Best Linear SVM parameters:")
print(grid_svm.best_params_)
print("Best Linear SVM CV score:", grid_svm.best_score_)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


/Users/carolinapires/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Linear SVM parameters:
{'clf__C': 1, 'clf__class_weight': 'balanced', 'tfidf__max_features': 10000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Best Linear SVM CV score: 0.7790396697742065


In [21]:
svm_test_metrics = evaluate_model(
    "Tuned Linear SVM",
    grid_svm.best_estimator_,
    X_test,
    y_test
)


Tuned Linear SVM
              model  accuracy  precision    recall  f1_weighted  f1_macro
0  Tuned Linear SVM  0.742772   0.742009  0.742772     0.741886  0.738308

Classification report:
              precision    recall  f1-score   support

       False       0.76      0.79      0.77       554
        True       0.73      0.68      0.70       449

    accuracy                           0.74      1003
   macro avg       0.74      0.74      0.74      1003
weighted avg       0.74      0.74      0.74      1003



In [22]:
tuned_results = pd.DataFrame([lr_test_metrics, svm_test_metrics])
tuned_results = tuned_results.sort_values(by="f1_macro", ascending=False)

display(tuned_results)

,model,accuracy,precision,recall,f1_weighted,f1_macro
1,Tuned Linear SVM,0.742772,0.742009,0.742772,0.741886,0.738308
0,Tuned Logistic Regression,0.737787,0.736990,0.737787,0.736925,0.733306


In [27]:
baseline_subset = results[
    (results["dataset"] == best_dataset_name) &
    (results["model"].isin(["Logistic Regression", "SVM"]))
].copy()

baseline_subset["model"] = baseline_subset["model"].replace({
    "Logistic Regression": "Baseline Logistic Regression",
    "SVM": "Baseline SVM"
})

baseline_subset["f1_weighted"] = np.nan
baseline_subset = baseline_subset[["model", "accuracy", "precision", "recall", "f1_weighted", "f1_macro"]]

comparison = pd.concat([baseline_subset, tuned_results], ignore_index=True)
comparison = comparison.sort_values(by="f1_macro", ascending=False)

display(comparison)

,model,accuracy,precision,recall,f1_weighted,f1_macro
0,Baseline Logistic Regression,0.791626,0.790617,0.791303,NaN,0.790883
1,Baseline SVM,0.790394,0.789381,0.789361,NaN,0.789371
2,Tuned Linear SVM,0.742772,0.742009,0.742772,0.741886,0.738308
3,Tuned Logistic Regression,0.737787,0.736990,0.737787,0.736925,0.733306


In [28]:
tuned_results.to_csv("../results/tuned_model_results.csv", index=False)

best_params_df = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "best_params": str(grid_lr.best_params_),
        "best_cv_score": grid_lr.best_score_
    },
    {
        "model": "Linear SVM",
        "best_params": str(grid_svm.best_params_),
        "best_cv_score": grid_svm.best_score_
    }
])

best_params_df.to_csv("../results/tuned_best_params.csv", index=False)

print("Saved:")
print("- ../results/tuned_model_results.csv")
print("- ../results/tuned_best_params.csv")

Saved:
- ../results/tuned_model_results.csv
- ../results/tuned_best_params.csv


Hyperparameter tuning did not improve model performance.